# Chapter 9: Advanced Concepts

In this chapter, we will go into some advanced linear algebra concepts. These will not come up all the time in quantum computing, but when they do, you should know what they are and where to find information about them. Almost all the topics are about decomposing a matrix. This becomes important in quantum computing because when we come up with a unitary transformation that we'd like to do on a quantum computer, we will only have certain unitary operators to use on it. Then, it becomes a question of which combination of available operators we should use.

In this chapter, we are going to cover the following main topics:

- Gram-Schmidt
- Cauchy-Schwarz and triangle inequalities
- Spectral decomposition
- Singular value decomposition
- Polar decomposition
- Operator functions and the matrix exponential

## Gram-Schmidt

The Gram-Schmidt process is an algorithm in which you input a basis set of vectors and it outputs a basis set that is orthogonal. We can then normalize that set to get an orthonormal basis.

> **Note:** The Gram-Schmidt process is also used in certain decompositions.

The algorithm:

1. Choose the first vector: $|v_1\rangle = |x_1\rangle$
2. For subsequent vectors:

$$|v_2\rangle = |x_2\rangle - \frac{\langle v_1|x_2\rangle}{\langle v_1|v_1\rangle}|v_1\rangle$$

$$|v_3\rangle = |x_3\rangle - \frac{\langle v_1|x_3\rangle}{\langle v_1|v_1\rangle}|v_1\rangle - \frac{\langle v_2|x_3\rangle}{\langle v_2|v_2\rangle}|v_2\rangle$$

3. Normalize each vector to get an orthonormal basis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Gram-Schmidt Process ---
def gram_schmidt(vectors):
    """Orthogonalize a set of vectors using Gram-Schmidt."""
    ortho = []
    for v in vectors:
        w = v.copy().astype(float)
        for u in ortho:
            w -= (np.dot(u, v) / np.dot(u, u)) * u
        ortho.append(w)
    # Normalize
    orthonormal = [u / np.linalg.norm(u) for u in ortho]
    return ortho, orthonormal

# Input basis (not orthogonal)
x1 = np.array([1, 1])
x2 = np.array([1, 0])

ortho, orthonormal = gram_schmidt([x1, x2])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original (non-orthogonal) basis
ax = axes[0]
ax.quiver(0, 0, x1[0], x1[1], angles='xy', scale_units='xy', scale=1, color='blue', linewidth=2, label=f'x₁ = {x1}')
ax.quiver(0, 0, x2[0], x2[1], angles='xy', scale_units='xy', scale=1, color='red', linewidth=2, label=f'x₂ = {x2}')
angle = np.degrees(np.arccos(np.dot(x1, x2) / (np.linalg.norm(x1) * np.linalg.norm(x2))))
ax.set_title(f'Original Basis\nAngle = {angle:.1f}° (not 90°)')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_aspect('equal')
ax.grid(True, alpha=0.3); ax.legend()
ax.axhline(y=0, color='k', linewidth=0.5); ax.axvline(x=0, color='k', linewidth=0.5)

# Orthogonalized (not yet normalized)
ax = axes[1]
for i, (v, c) in enumerate(zip(ortho, ['blue', 'red'])):
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color=c, linewidth=2, 
              label=f'v{i+1} = [{v[0]:.2f}, {v[1]:.2f}]')
angle2 = np.dot(ortho[0], ortho[1])
ax.set_title(f'After Gram-Schmidt\nDot product = {angle2:.6f} (orthogonal!)')
ax.set_xlim(-0.7, 1.5); ax.set_ylim(-0.7, 1.5); ax.set_aspect('equal')
ax.grid(True, alpha=0.3); ax.legend()
ax.axhline(y=0, color='k', linewidth=0.5); ax.axvline(x=0, color='k', linewidth=0.5)

# Orthonormalized
ax = axes[2]
for i, (v, c) in enumerate(zip(orthonormal, ['blue', 'red'])):
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color=c, linewidth=2,
              label=f'ê{i+1} = [{v[0]:.3f}, {v[1]:.3f}]')
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.5)
ax.add_patch(circle)
ax.set_title(f'Orthonormal Basis\n||ê₁|| = {np.linalg.norm(orthonormal[0]):.1f}, ||ê₂|| = {np.linalg.norm(orthonormal[1]):.1f}')
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect('equal')
ax.grid(True, alpha=0.3); ax.legend()
ax.axhline(y=0, color='k', linewidth=0.5); ax.axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

# Cauchy-Schwarz inequality
x = np.array([1, 2, 3])
y = np.array([4, -1, 2])
lhs = abs(np.dot(x, y))
rhs = np.linalg.norm(x) * np.linalg.norm(y)
print(f"Cauchy-Schwarz:  |⟨x|y⟩| = {lhs:.4f}  ≤  ||x||·||y|| = {rhs:.4f}  →  {lhs <= rhs + 1e-10}")

# Triangle inequality
lhs_t = np.linalg.norm(x + y)
rhs_t = np.linalg.norm(x) + np.linalg.norm(y)
print(f"Triangle ineq:   ||x+y|| = {lhs_t:.4f}  ≤  ||x||+||y|| = {rhs_t:.4f}  →  {lhs_t <= rhs_t + 1e-10}")

## Cauchy-Schwarz and Triangle Inequalities

### Cauchy-Schwarz inequality

One of the most important inequalities in mathematics. The absolute value of the inner product of two vectors is less than or equal to the product of their norms. They are only equal if the two vectors are linearly dependent:

$$|\langle x|y\rangle| \leq \|x\| \cdot \|y\|$$

### Triangle inequality

The length of two sides of a triangle must always be more than (or equal to, in the degenerate case) the length of one side:

$$\|x + y\| \leq \|x\| + \|y\|$$

> *Figure 9.1 – Three example triangles for the triangle inequality*

## Spectral Decomposition

The spectrum of a square matrix is the set of its eigenvalues. All matrices representing a linear operator have the same spectrum.

### Diagonal matrices

A diagonal matrix has zero on all entries outside the main diagonal. Cool features:
1. All eigenvalues are on the main diagonal
2. Very easy to exponentiate

### Spectral theory

The spectral theorem tells us when an operator can be represented by a diagonal matrix (**diagonalizable**). A diagonalizable matrix A can be factored as:

$$A = PDP^{-1}$$

For normal matrices:

$$A = U\Lambda U^\dagger$$

where U is a unitary matrix (columns are eigenvectors of A) and Λ is a diagonal matrix with eigenvalues on its main diagonal.

This means any normal matrix can be decomposed into a unitary matrix, its conjugate transpose, and a diagonal matrix of eigenvalues.

### Bra-ket notation

The spectral decomposition in bra-ket notation:

$$A = \sum_i \lambda_i |v_i\rangle\langle v_i|$$

Any normal operator can be represented as a **linear combination of outer products** composed of just its eigenvalues and eigenvectors!

### Example: Y gate

$$Y = \begin{bmatrix} 0 & -i \\ i & 0 \end{bmatrix}$$

Eigenvalues: 1 and -1. The eigenvectors are the |i⟩ and |-i⟩ states:

$$Y = 1 \cdot |i\rangle\langle i| + (-1) \cdot |-i\rangle\langle -i| \quad \text{(1)}$$

In [ ]:
import numpy as np

# --- Spectral Decomposition ---
# Pauli-Y gate
Y = np.array([[0, -1j], [1j, 0]])
eigenvalues, eigenvectors = np.linalg.eigh(Y)

print("=== Spectral Decomposition of Pauli-Y ===")
print(f"Y =\n{Y}\n")
print(f"Eigenvalues: {eigenvalues}")
for i in range(len(eigenvalues)):
    print(f"  λ_{i+1} = {eigenvalues[i]:.1f}, eigenvector = {np.round(eigenvectors[:, i], 4)}")

# Reconstruct: A = Σ λ_i |v_i⟩⟨v_i|
Y_reconstructed = np.zeros_like(Y)
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i:i+1]
    Y_reconstructed += eigenvalues[i] * (v @ v.conj().T)

print(f"\nReconstructed from spectral decomposition:")
print(f"Y = Σ λᵢ|vᵢ⟩⟨vᵢ| =\n{np.round(Y_reconstructed, 6)}")
print(f"Matches original? {np.allclose(Y, Y_reconstructed)}\n")

# Diagonalization: A = U Λ U†
U = eigenvectors
Lambda = np.diag(eigenvalues)
print(f"U (eigenvectors as columns) =\n{np.round(U, 4)}\n")
print(f"Λ (diagonal eigenvalues) =\n{Lambda}\n")
print(f"U Λ U† =\n{np.round(U @ Lambda @ U.conj().T, 6)}")
print(f"Matches Y? {np.allclose(Y, U @ Lambda @ U.conj().T)}")

# A general Hermitian matrix
A = np.array([[4, 1-1j], [1+1j, 3]])
evals, evecs = np.linalg.eigh(A)
print(f"\n=== Hermitian Matrix Spectral Decomposition ===")
print(f"A =\n{A}")
print(f"Eigenvalues (all real!): {np.round(evals, 4)}")
A_recon = evecs @ np.diag(evals) @ evecs.conj().T
print(f"Reconstructed matches? {np.allclose(A, A_recon)}")

## Singular Value Decomposition

SVD is probably the most famous decomposition. It is at the core of search engines and machine learning algorithms. It can be used on **any** type of matrix, even rectangular ones.

For any matrix A:

$$A = U\Sigma V^\dagger$$

where U is a unitary matrix, Σ is a diagonal matrix with **singular values** on its diagonal, and V is also a unitary matrix. This decomposition is not unique.

The singular values are found by taking the square root of the eigenvalues of $AA^\dagger$.

In [ ]:
import numpy as np

# --- Singular Value Decomposition (SVD) ---
A = np.array([[1, 2], [3, 4], [5, 6]])  # rectangular matrix (3x2)

U, sigma, Vd = np.linalg.svd(A, full_matrices=True)

print("=== Singular Value Decomposition ===")
print(f"A (3×2) =\n{A}\n")
print(f"U (3×3, unitary) =\n{np.round(U, 4)}\n")
print(f"Singular values σ = {np.round(sigma, 4)}\n")
print(f"V† (2×2, unitary) =\n{np.round(Vd, 4)}\n")

# Reconstruct: A = U Σ V†
Sigma_full = np.zeros_like(A, dtype=float)
np.fill_diagonal(Sigma_full, sigma)
A_reconstructed = U @ Sigma_full @ Vd
print(f"Reconstructed A = U Σ V† =\n{np.round(A_reconstructed, 6)}")
print(f"Matches original? {np.allclose(A, A_reconstructed)}\n")

# Verify U and V are unitary
print(f"U†U ≈ I? {np.allclose(U.conj().T @ U, np.eye(U.shape[0]))}")
print(f"V†V ≈ I? {np.allclose(Vd @ Vd.conj().T, np.eye(Vd.shape[0]))}")

# Singular values from eigenvalues of A†A
eigvals_ATA = np.linalg.eigvalsh(A.conj().T @ A)
print(f"\nEigenvalues of A†A: {np.round(sorted(eigvals_ATA, reverse=True), 4)}")
print(f"√(eigenvalues)    : {np.round(np.sqrt(sorted(eigvals_ATA, reverse=True)), 4)}")
print(f"Singular values   : {np.round(sigma, 4)}")

# --- Polar Decomposition ---
# For a square matrix: A = U P  where U is unitary, P is positive semi-definite
B = np.array([[1, 1j], [0, 1+1j]])
U_svd, s, Vd_svd = np.linalg.svd(B)
# P = V Σ V†, U_polar = U_svd @ Vd_svd
P = Vd_svd.conj().T @ np.diag(s) @ Vd_svd
U_polar = U_svd @ Vd_svd

print(f"\n=== Polar Decomposition: B = U·P ===")
print(f"B =\n{np.round(B, 4)}\n")
print(f"U (unitary) =\n{np.round(U_polar, 4)}\n")
print(f"P (positive semi-definite Hermitian) =\n{np.round(P, 4)}\n")
print(f"U·P =\n{np.round(U_polar @ P, 4)}")
print(f"Matches B? {np.allclose(B, U_polar @ P)}")
print(f"P is Hermitian? {np.allclose(P, P.conj().T)}")
print(f"Eigenvalues of P (non-negative): {np.round(np.linalg.eigvalsh(P), 4)}")

## Polar Decomposition

Polar decomposition allows you to factor any matrix into unitary and positive semi-definite Hermitian matrices. It can be seen as breaking down a linear transformation into a rotation/reflection and scaling in ℝⁿ:

$$A = UP$$

where U is a unitary matrix and P is a positive semi-definite matrix.

In [ ]:
import numpy as np
from scipy.linalg import expm

# --- Matrix Exponential ---
# Simple case: diagonal matrix (Z gate)
Z = np.array([[1, 0], [0, -1]])
exp_Z = expm(Z)

print("=== Matrix Exponential ===")
print(f"Z =\n{Z}\n")
print(f"exp(Z) =\n{np.round(exp_Z, 6)}")
print(f"Expected: diag(e, e⁻¹) = diag({np.e:.6f}, {1/np.e:.6f})\n")

# Non-diagonal: Pauli-X
X = np.array([[0, 1], [1, 0]], dtype=complex)
exp_X = expm(X)
print(f"X =\n{X}\n")
print(f"exp(X) =\n{np.round(exp_X, 6)}\n")

# Verify via diagonalization: exp(A) = P exp(D) P⁻¹
evals, P = np.linalg.eig(X)
exp_D = np.diag(np.exp(evals))
exp_X_manual = P @ exp_D @ np.linalg.inv(P)
print(f"Via diagonalization: P·exp(D)·P⁻¹ =\n{np.round(exp_X_manual, 6)}")
print(f"Matches scipy expm? {np.allclose(exp_X, exp_X_manual)}\n")

# Rotation gate: R_z(θ) = exp(-iθZ/2) — common in quantum computing
theta = np.pi / 4
Rz = expm(-1j * theta / 2 * Z)
print(f"=== Quantum Rotation Gate ===")
print(f"R_z(π/4) = exp(-iπ/8 · Z) =\n{np.round(Rz, 6)}")
print(f"Expected: diag(e^(-iπ/8), e^(iπ/8))")
print(f"  e^(-iπ/8) = {np.exp(-1j*np.pi/8):.6f}")
print(f"  e^(+iπ/8) = {np.exp(1j*np.pi/8):.6f}")
print(f"Unitary? {np.allclose(Rz.conj().T @ Rz, np.eye(2))}")

## Operator Functions and the Matrix Exponential

What about functions like sin(A) where A is a matrix? Mathematicians have made this well-defined.

If a matrix A is diagonalizable:

$$A = PDP^{-1} \quad \text{(2)}$$

Then:

$$f(A) = P \begin{bmatrix} f(d_1) & 0 & \cdots \\ 0 & f(d_2) & \cdots \\ \vdots & \vdots & \ddots \end{bmatrix} P^{-1} \quad \text{(3)}$$

The function is evaluated for every value on the main diagonal of D.

### The matrix exponential

A function that comes up a lot in quantum computing:

$$e^A = \exp(A)$$

For a diagonal matrix like the Z operator, this is easy to calculate:

$$\exp\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} = \begin{pmatrix} e & 0 \\ 0 & e^{-1} \end{pmatrix}$$

For non-diagonal matrices, first diagonalize, then compute the exponential on the diagonal, then multiply back.

## Summary

That wraps up this chapter. We've covered wonderful decompositions (spectral, SVD, polar) and advanced concepts like operator functions and the matrix exponential. This chapter also concludes the book. Take this math and go forth to infinity and beyond in the universe of quantum computing!